# ATLOP on DocRED

Learn a threshold for each entity pair, then pool the context both entities attend to.

In [1]:
from pathlib import Path
import json
import sys
import warnings
warnings.filterwarnings("ignore", message="IProgress not found.*")
import torch

module = Path.cwd() if Path("src").exists() else Path("ie_atlop")
sys.path.insert(0, str((module / "src").resolve()))
from infrastructure.losses import ATLoss

## Adaptive thresholding — class zero is the learned threshold.

In [2]:
logits = torch.tensor([[2., 3., 1., 4.], [5., 3., 1., 4.]], requires_grad=True)
labels = torch.tensor([[0., 1., 0., 1.], [1., 0., 0., 0.]])
predictions = ATLoss().get_label(logits, num_labels=4)
print("Logits:", logits.detach().tolist())
print("Predictions:", predictions.tolist())
assert predictions.tolist() == [[0., 1., 0., 1.], [1., 0., 0., 0.]]
loss = ATLoss()(logits, labels.clone())
loss.backward()
assert torch.isfinite(logits.grad).all()
print("Loss:", loss.item())

Logits: [[2.0, 3.0, 1.0, 4.0], [5.0, 3.0, 1.0, 4.0]]
Predictions: [[0.0, 1.0, 0.0, 1.0], [1.0, 0.0, 0.0, 0.0]]
Loss: 1.274095058441162


## Localized context — multiply the two attention distributions and normalize.

In [3]:
head_attention = torch.tensor([0.1, 0.7, 0.2])
tail_attention = torch.tensor([0.2, 0.6, 0.2])
shared_attention = head_attention * tail_attention
shared_attention = shared_attention / (shared_attention.sum() + 1e-5)
token_states = torch.tensor([[1., 0.], [0., 1.], [1., 1.]])
context = shared_attention @ token_states
print("Head:", head_attention.tolist())
print("Tail:", tail_attention.tolist())
print("Shared:", shared_attention.tolist())
print("Context:", context.tolist())

Head: [0.10000000149011612, 0.699999988079071, 0.20000000298023224]
Tail: [0.20000000298023224, 0.6000000238418579, 0.20000000298023224]
Shared: [0.04166579991579056, 0.8749817609786987, 0.08333159983158112]
Context: [0.12499739974737167, 0.9583133459091187]


## Official files — count the supplied documents.

In [4]:
for name in ["train_annotated", "dev", "test"]:
    documents = json.loads((module / "data/docred" / (name + ".json")).read_text())
    print(name, len(documents), "documents;", sum("labels" in row for row in documents), "labeled")


train_annotated 3053 documents; 3053 labeled
dev 998 documents; 998 labeled
test 1000 documents; 0 labeled


## Training configuration — `make train` saves the best dev checkpoint.

In [5]:
print((module / "scripts" / "train.sh").read_text())
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

#!/usr/bin/env bash
set -euo pipefail
cd "$(dirname "$0")/.."
export PATH="$(cd .. && pwd)/.venv/bin:$PATH"
export PYTHONPATH="$PWD/src"
export PYTHONUNBUFFERED=1
mkdir -p checkpoints artifacts
python -m bootstrap.train --data_dir ./data/docred \
--transformer_type bert \
--model_name_or_path checkpoints/bert-base-cased \
--train_file train_annotated.json \
--dev_file dev.json \
--test_file test.json \
--train_batch_size 4 \
--test_batch_size 8 \
--gradient_accumulation_steps 1 \
--num_labels 4 \
--learning_rate 5e-5 \
--max_grad_norm 1.0 \
--warmup_ratio 0.06 \
--num_train_epochs 30.0 \
--seed 66 \
--num_class 97 \
--save_path checkpoints/local-bert.pt 2>&1 | tee artifacts/train.log
python src/interfaces/compare.py artifacts/train.log --completed > artifacts/comparison.tmp
mv artifacts/comparison.tmp results.json
cat results.json

PyTorch: 1.13.1+cu117 CUDA: 11.7
CUDA available: True


In [6]:
try:
    sample = torch.ones((2, 2), device="cuda")
    print("CUDA matmul:", (sample @ sample).cpu().tolist())
except RuntimeError as error:
    print("CUDA execution failed:", str(error))

CUDA matmul: [[2.0, 2.0], [2.0, 2.0]]


## Same-split comparison — select Ign F1 from the evaluation with the best dev F1.

In [7]:
print((module / "results.json").read_text())


{
  "split": "dev",
  "seed": 66,
  "training_completed": true,
  "evaluations": 30,
  "best_epoch": 29,
  "selection": "maximum dev_F1; Ign F1 from the same evaluation",
  "measured": {
    "dev_F1": 60.80034987972884,
    "dev_F1_ign": 58.97446685786348
  },
  "paper_five_seed_mean": {
    "dev_F1": 61.09,
    "dev_F1_ign": 59.22
  },
  "difference_percentage_points": {
    "dev_F1": -0.2896501202711619,
    "dev_F1_ign": -0.2455331421365159
  },
  "test": "unscored; requires official evaluation of artifacts/result.json"
}



## Weaknesses

| Weakness | What happens | Fix |
|---|---|---|
| Released dev split differs from paper count | Official download has 998 documents; Table 1 lists 1,000. | Keep the official release; report the discrepancy. |
| Test labels are withheld | Downloaded test file has no labeled documents. | Obtain official scores for the generated submission. |
